In [0]:
%run ./_ddl

DataFrame[]

In [0]:
import requests
import json
import logging
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

In [0]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger("download_competitions")

###### miscellaneous

In [0]:
# ### Read Matches Dictionary

# with open("/Volumes/workspace/football_project/docs/matches_dict.json", "r") as file:
#     data = json.load(file)
#     matches_dict = json.dumps(data, indent=4)

# print(matches_dict)

{
    "9": "1. Bundesliga",
    "1267": "African Cup of Nations",
    "16": "Champions League",
    "223": "Copa America",
    "87": "Copa del Rey",
    "37": "FA Women's Super League",
    "1470": "FIFA U20 World Cup",
    "43": "FIFA World Cup",
    "135": "Frauen Bundesliga",
    "1238": "Indian Super league",
    "11": "La Liga",
    "182": "Liga F",
    "81": "Liga Profesional",
    "7": "Ligue 1",
    "44": "Major League Soccer",
    "116": "North American League",
    "49": "NWSL",
    "2": "Premier League",
    "12": "Serie A",
    "131": "Serie A Women",
    "55": "UEFA Euro",
    "35": "UEFA Europa League",
    "53": "UEFA Women's Euro",
    "72": "Women's World Cup"
}


##### Functions

In [0]:
### Get Seasons by Competition ID

def get_competition_seasons(competition_id: int) -> list[dict]:

    comp_url = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json"
    response = requests.get(comp_url)

    # if not found competitions.json
    if response.status_code != 200:
        raise Exception(f"Failed to fetch competitions.json: {response.status_code}")
    
    competitions = response.json()
    seasons = [
        {"competition_name": c["competition_name"],
         "season_id": c["season_id"],
         "season_name": c["season_name"]}
         for c in competitions if c["competition_id"] == competition_id
        ]
    
    # if comp is none
    if not seasons:
        raise ValueError(f"Competition ID {competition_id} not found in competitions.json")
    
    # if comp found | success
    comp_name = seasons[0]['competition_name']
    logger.info(f"Found '{comp_name}' (ID: {competition_id}) with {len(seasons)} season(s)")

    return seasons

In [0]:
### Download [LE360]: Lineups, Events, and 360 Files

def download_LE360(match_ids: list, 
                   volume_base: str, 
                   max_workers: int = 20, 
                   force_refresh: bool = False
                   ) -> tuple[int, int, int]:
    
        """
        Downloads lineups, events, and 360 files in parallel for all given match_ids.

        Self-Healing Logic:
        - If force_refresh=False and file exists (>0 bytes) -> SKIP.
        - If file is missing or 0 bytes -> DOWNLOAD (Self-Heal).
        - If force_refresh=True -> OVERWRITE everything.
        """
        for folder in ["matches", "events", "lineups", "three_sixty"]:
            os.makedirs(f"{volume_base}/{folder}", exist_ok=True)
        
        def _download_single_file(volume_folder: str, match_id: str) -> tuple[str, str, str]:
            git_folder = "three-sixty" if volume_folder == "three_sixty" else volume_folder
            
            git_url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/{git_folder}/{match_id}.json"
            save_dir = f"{volume_base}/{volume_folder}/{match_id}.json"

            if not force_refresh and os.path.exists(save_dir) and os.path.getsize(save_dir) > 0:
                return ("skipped", volume_folder, match_id)

            try:
                r = requests.get(git_url, timeout=15)
                if r.status_code == 200:
                    with open(save_dir, "w") as file:
                        json.dump(r.json(), file)
                    return ("saved", volume_folder, match_id)
                elif r.status_code == 404:
                    return ("not_found", volume_folder, match_id)
                else:
                    logger.warning(f"HTTP {r.status_code} on {git_url}")
                    return ("error", volume_folder, match_id)

            except Exception as e:
                logger.error(f"Network/IO Error on {git_url}: {e}")
                return ("error", volume_folder, match_id)

        data_folders = ["events", "lineups", "three_sixty"]
        logger.info(f"Starting ThreadPoolExecutor with {max_workers} workers...")

        missing_by_folder = {"events": 0, "lineups": 0, "three_sixty": 0}
        downloaded_count = 0
        skipped_count = 0
        error_count = 0
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(_download_single_file, folder, str(m_id))
                for m_id in match_ids
                for folder in data_folders
            ]

            for future in as_completed(futures):
                try:
                    status, folder_name, match_id = future.result()

                    if status == "saved":
                        downloaded_count += 1
                    elif status == "skipped":
                        skipped_count += 1
                    elif status == "not_found":
                        missing_by_folder[folder_name] += 1
                    elif status == "error":
                        error_count += 1
                        
                except Exception as e:
                    error_count += 1
                    logger.error(f"Worker thread crashed: {e}")

        logger.info(
            f"=== Missing Report === "
            f"Events missing: {missing_by_folder['events']} | "
            f"Lineups missing: {missing_by_folder['lineups']} | "
            f"360 missing: {missing_by_folder['three_sixty']} (Normal for pre-2022)"
        )

        return downloaded_count, skipped_count, error_count

In [0]:
### Donwload Competition: Orchestration

def download_competition_pipeline(
    competition_id: int, 
    volume_base: str = "/Volumes/workspace/football_project/raw_data", 
    max_workers: int = 20, 
    force_refresh: bool = False
    ) -> None:

    """
    MASTER PIPELINE:
    1. Calls get_competition_seasons()
    2. Collects match_ids > Downloads match fixtures
    3. Calls download_LE360()
    """
    logger.info(f"==================================================")
    logger.info(f"Starting Ingestion Pipeline for Competition: {competition_id}")
    logger.info(f"==================================================")

    ### Get Seasons
    seasons = get_competition_seasons(competition_id)

    ### Get Match IDs > Download to Volume: Matches [Competition_ID/Season_ID.json]
    all_match_ids = set()
    GITHUB_RAW_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches"

    match_dir = f"{volume_base}/matches/{competition_id}"
    os.makedirs(match_dir, exist_ok=True)
    
    for s in seasons:
        s_id = s["season_id"]
        s_name = s["season_name"]
        save_dir = f"{match_dir}/{s_id}.json"

        # Local Cache
        if not force_refresh and os.path.exists(save_dir) and os.path.getsize(save_dir) > 0:
            with open(save_dir, "r") as f:
                seasons_data = json.load(f)
            for sd in seasons_data:
                all_match_ids.add(str(sd["match_id"]))
            logger.info(f"Season {s_name}: {len(seasons_data)} matches (Cached)")
            continue
        
        # If not cache, Fetch from GitHub           
        seasons_url = f"{GITHUB_RAW_URL}/{competition_id}/{s_id}.json"
        response = requests.get(seasons_url)

        if response.status_code == 200:
            seasons_data = response.json()

            with open(save_dir, "w") as f:
                json.dump(seasons_data, f)

            for sd in seasons_data:
                all_match_ids.add(str(sd["match_id"]))
            logger.info(f"Season {s_name}: {len(seasons_data)} matches")

        else:
            logger.warning(f"Failed to fetch matches for season {s_name}")

    logger.info(f"Total Unique Matches: {len(all_match_ids)}")

    ### Load Matches LE360
    total_saved, total_skipped, total_errors = download_LE360(list(all_match_ids),volume_base, max_workers, force_refresh=force_refresh)

    logger.info(
        f"Pipeline Complete for {seasons[0]['competition_name']} | "
        f"Downloaded: {total_saved} | Skipped: {total_skipped} | Errors: {total_errors}" 
        )


##### Run

In [0]:
### Interested List

# - "9": "1. Bundesliga",
# - "16": "Champions League",
# - "1470": "FIFA U20 World Cup",/Volumes/workspace/football_project/matches/43/106.json
# - "11": "La Liga",
# - "2": "Premier League",
# - "55": "UEFA Euro",
# - "35": "UEFA Europa League",
# - "53": "UEFA Women's Euro",
# - "72": "Women's World Cup"

download_competition_pipeline(43) #43: World Cup

2026-08-18 08:36:31 [INFO] ==================================================
2026-08-18 08:36:31 [INFO] Starting Ingestion Pipeline for Competition: 43
2026-08-18 08:36:31 [INFO] ==================================================
2026-08-18 08:36:31 [INFO] Found 'FIFA World Cup' (ID: 43) with 8 season(s)
2026-08-18 08:36:32 [INFO] Season 2022: 64 matches
2026-08-18 08:36:32 [INFO] Season 2018: 64 matches
2026-08-18 08:36:32 [INFO] Season 1990: 1 matches
2026-08-18 08:36:32 [INFO] Season 1986: 3 matches
2026-08-18 08:36:33 [INFO] Season 1974: 6 matches
2026-08-18 08:36:33 [INFO] Season 1970: 6 matches
2026-08-18 08:36:33 [INFO] Season 1962: 1 matches
2026-08-18 08:36:33 [INFO] Season 1958: 2 matches
2026-08-18 08:36:33 [INFO] Total Unique Matches: 147
2026-08-18 08:36:33 [INFO] Starting ThreadPoolExecutor with 20 workers...
2026-08-18 08:36:38 [INFO] === Missing Report === Events missing: 0 | Lineups missing: 0 | 360 missing: 0 (Normal for pre-2022)
2026-08-18 08:36:38 [INFO] Pipeline 